# Differentiable Parisi Solver

This notebook demonstrates the full power of ParisiJax's variational solver:
1. RS limitations below T_c
2. 1RSB improvement
3. Full k-RSB optimization
4. Convergence diagnostics
5. Ground state energy validation
6. Differentiating through the Parisi PDE

**Runtime:** < 5 min on CPU

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from parisijax.core.solver import (
    ground_state_energy,
    high_temp_free_energy,
    one_rsb_free_energy,
    optimize_parisi,
    optimize_parisi_multistart,
    parisi_free_energy,
    rs_free_energy,
)

## 1. RS vs High-Temperature Approximation

In [ ]:
betas = jnp.linspace(0.2, 3.0, 30)
f_rs = [float(rs_free_energy(b)) for b in betas]
f_ht = [float(high_temp_free_energy(b)) for b in betas]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.array(betas), f_rs, 'o-', label='RS', markersize=4)
ax.plot(np.array(betas), f_ht, '--', label=r'High-$T$ approx', alpha=0.7)
ax.axvline(1.0, color='red', ls=':', alpha=0.5, label=r'$\beta_c$')
ax.set_xlabel(r'$\beta$'); ax.set_ylabel('Free energy')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title('RS breaks down below T_c')
plt.tight_layout(); plt.show()

## 2. 1RSB Improvement

In [ ]:
beta = 1.5
f_rs_val = float(rs_free_energy(beta))

# Grid search for best 1RSB parameters
best_f_1rsb = float('inf')
best_params = None
for q0 in np.linspace(0.05, 0.4, 8):
    for q1 in np.linspace(0.5, 0.95, 8):
        for m in np.linspace(0.1, 0.9, 8):
            f = float(one_rsb_free_energy(q0, q1, m, beta))
            if f < best_f_1rsb:
                best_f_1rsb = f
                best_params = (q0, q1, m)

print(f"RS free energy at beta={beta}: {f_rs_val:.6f}")
print(f"Best 1RSB free energy:        {best_f_1rsb:.6f}")
print(f"1RSB improvement:             {f_rs_val - best_f_1rsb:.6f}")
print(f"Best 1RSB params: q0={best_params[0]:.2f}, q1={best_params[1]:.2f}, m={best_params[2]:.2f}")

## 3. Full k-RSB Optimization with Convergence Diagnostics

In [ ]:
result = optimize_parisi(beta=2.0, k=10, n_steps=2000, lr=0.01, seed=0)

print(f"Free energy: {result.free_energy:.6f}")
print(f"Converged: {result.converged}")
print(f"Steps used: {result.n_steps_used}")
print(f"Final grad norm: {result.grad_norm_final:.2e}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(np.array(result.loss_history))
ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
ax1.set_title('Optimization convergence')
ax1.grid(True, alpha=0.3)

# Plot the Parisi function q(x)
q_opt = np.array(result.q)
m_opt = np.array(result.m)
q_ext = np.concatenate([[0.0], q_opt])
m_ext = np.concatenate([[0.0], m_opt])
for i in range(len(q_opt)):
    ax2.hlines(q_ext[i+1], m_ext[i], m_ext[i+1], colors='blue', lw=2)
    if i < len(q_opt) - 1:
        ax2.vlines(m_ext[i+1], q_ext[i+1], q_ext[i+2], colors='blue', lw=2, ls='--')
ax2.set_xlabel('x'); ax2.set_ylabel('q(x)')
ax2.set_title(f'Parisi function at beta={2.0}')
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 4. Ground State Energy Validation

In [ ]:
# This may take a minute
result_gs = optimize_parisi_multistart(beta=10.0, k=20, n_steps=3000, lr=0.005, n_starts=3)

known_E0 = ground_state_energy()
print(f"Known ground state energy:   {known_E0:.4f}")
print(f"ParisiJax result:            {result_gs.free_energy:.4f}")
print(f"Absolute error:              {abs(result_gs.free_energy - known_E0):.4f}")

## 5. Differentiating Through the Parisi PDE

In [ ]:
# Compute gradients of the free energy w.r.t. the RSB parameters
key = jax.random.PRNGKey(0)
k = 5
q_raw = jax.random.normal(key, (k,)) * 0.1
m_raw = jax.random.normal(jax.random.split(key)[0], (k,)) * 0.1

grad_fn = jax.grad(parisi_free_energy, argnums=(0, 1))
gq, gm = grad_fn(q_raw, m_raw, 1.5)

print("Gradient w.r.t. q_raw:", np.array(gq))
print("Gradient w.r.t. m_raw:", np.array(gm))
print(f"All finite: {bool(jnp.all(jnp.isfinite(gq)) & jnp.all(jnp.isfinite(gm)))}")

## 6. Cosine LR Schedule Comparison

In [ ]:
r_const = optimize_parisi(beta=2.0, k=10, n_steps=1000, lr=0.01, lr_schedule='constant', seed=0)
r_cosine = optimize_parisi(beta=2.0, k=10, n_steps=1000, lr=0.01, lr_schedule='cosine', seed=0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.array(r_const.loss_history), label='Constant LR', alpha=0.8)
ax.plot(np.array(r_cosine.loss_history), label='Cosine LR', alpha=0.8)
ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.set_title('LR Schedule Comparison')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Constant: f = {r_const.free_energy:.6f}")
print(f"Cosine:   f = {r_cosine.free_energy:.6f}")